In [1]:
import pandas as pd
import numpy as np

csv_path = r"C:\Users\boses\Desktop\bfsi-fraud-project\data\UPI_FRAUD_100K.csv"
df = pd.read_csv(csv_path)

print("Shape:", df.shape)
print("Fraud rate:", round(df['fraud'].mean(), 5))

# Parse Date + Time into a single datetime column
df['datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%d/%m/%y %I:%M:%S %p'
)

print("\nDatetime sample:")
print(df[['Date', 'Time', 'datetime']].head())
print("\nDatetime range:", df['datetime'].min(), "to", df['datetime'].max())
print("Any NaT:", df['datetime'].isna().sum())

Shape: (100000, 20)
Fraud rate: 0.23986

Datetime sample:
       Date         Time            datetime
0  12/02/22  08:20:00 AM 2022-02-12 08:20:00
1  26/03/23  02:28:00 PM 2023-03-26 14:28:00
2  10/05/22  07:24:00 PM 2022-05-10 19:24:00
3  14/12/23  10:18:00 AM 2023-12-14 10:18:00
4  28/06/23  05:06:00 AM 2023-06-28 05:06:00

Datetime range: 2022-01-01 00:01:00 to 2023-12-31 23:46:00
Any NaT: 0


In [2]:
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek  # 0=Mon, 6=Sun
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

print("Hour distribution:")
print(df['hour'].value_counts().sort_index())
print("\nDay-of-week distribution:")
print(df['day_of_week'].value_counts().sort_index())

Hour distribution:
hour
0     4291
1     4126
2     4089
3     4177
4     4085
5     4139
6     4351
7     4076
8     4164
9     4298
10    4231
11    4127
12    4175
13    4202
14    4223
15    4082
16    4046
17    4221
18    4185
19    4212
20    4147
21    4056
22    4130
23    4167
Name: count, dtype: int64

Day-of-week distribution:
day_of_week
0    14206
1    14167
2    14334
3    14453
4    14285
5    14217
6    14338
Name: count, dtype: int64


C:\Users\boses\AppData\Local\Temp\ipykernel_17180\2029887101.py:2: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['day_of_week'] = df['datetime'].dt.dayofweek  # 0=Mon, 6=Sun


In [3]:
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.day_of_week   # corrected name
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

print("Hour distribution range:", df['hour'].value_counts().min(), "to", df['hour'].value_counts().max())
print("Day-of-week distribution range:", df['day_of_week'].value_counts().min(), "to", df['day_of_week'].value_counts().max())
print("Weekend count:", int(df['is_weekend'].sum()))

Hour distribution range: 4046 to 4351
Day-of-week distribution range: 14167 to 14453
Weekend count: 28555


In [4]:
import numpy as np

df['amount_log'] = np.log1p(df['amount'])

numeric_features = [
    'Transaction_Frequency',
    'Transaction_Amount_Deviation',
    'Days_Since_Last_Transaction',
    'amount_log',
    'hour',
    'day_of_week',
    'is_weekend',
]

categorical_features = [
    'Transaction_Type',
    'Payment_Gateway',
    'Merchant_Category',
    'Transaction_Channel',
    'Device_OS',
    'Transaction_Status',
]

print("Null counts in numeric features:")
print(df[numeric_features].isnull().sum())
print("\nNull counts in categorical features:")
print(df[categorical_features].isnull().sum())

df_encoded = pd.get_dummies(df[categorical_features], drop_first=False).astype(int)

print("\nEncoded categorical shape:", df_encoded.shape)
print("Encoded columns:", list(df_encoded.columns))

Null counts in numeric features:
Transaction_Frequency           0
Transaction_Amount_Deviation    0
Days_Since_Last_Transaction     0
amount_log                      0
hour                            0
day_of_week                     0
is_weekend                      0
dtype: int64

Null counts in categorical features:
Transaction_Type       0
Payment_Gateway        0
Merchant_Category      0
Transaction_Channel    0
Device_OS              0
Transaction_Status     0
dtype: int64

Encoded categorical shape: (100000, 36)
Encoded columns: ['Transaction_Type_Bank Transfer', 'Transaction_Type_Bill Payment', 'Transaction_Type_Investment', 'Transaction_Type_Other', 'Transaction_Type_Purchase', 'Transaction_Type_Refund', 'Transaction_Type_Subscription', 'Payment_Gateway_Alpha Bank', 'Payment_Gateway_Bank of Data', 'Payment_Gateway_CReditPAY', 'Payment_Gateway_Dummy Bank', 'Payment_Gateway_Gamma Bank', 'Payment_Gateway_Other', 'Payment_Gateway_SamplePay', 'Payment_Gateway_Sigma Bank', 'Payment

In [5]:
X = pd.concat([df[numeric_features], df_encoded], axis=1)
y = df['fraud']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("y distribution:")
print(y.value_counts())
print("\nFeature count:", X.shape[1])
print("Any nulls in X:", X.isnull().sum().sum())
print("Any infs in X:", np.isinf(X.values).sum())
print("\nFirst 5 feature names:", list(X.columns)[:5])
print("Last 5 feature names:", list(X.columns)[-5:])

X shape: (100000, 43)
y shape: (100000,)
y distribution:
fraud
0    76014
1    23986
Name: count, dtype: int64

Feature count: 43
Any nulls in X: 0
Any infs in X: 0

First 5 feature names: ['Transaction_Frequency', 'Transaction_Amount_Deviation', 'Days_Since_Last_Transaction', 'amount_log', 'hour']
Last 5 feature names: ['Device_OS_Windows', 'Device_OS_iOS', 'Transaction_Status_Completed', 'Transaction_Status_Failed', 'Transaction_Status_Pending']


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain fraud rate:", round(y_train.mean(), 5))
print("Test fraud rate :", round(y_test.mean(), 5))

Train shape: (80000, 43)
Test shape: (20000, 43)

Train fraud rate: 0.23986
Test fraud rate : 0.23985


In [8]:
from sklearn.linear_model import LogisticRegression
import time

start = time.time()
model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
)
model.fit(X_train, y_train)
print(f"Trained in {time.time() - start:.2f} seconds.")
print("Number of coefficients:", len(model.coef_[0]))

Trained in 0.59 seconds.
Number of coefficients: 43


In [9]:
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score,
)

y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion matrix (threshold=0.5):")
print(f"                   Predicted 0    Predicted 1")
print(f"  Actual 0 (TN/FP)  {tn:>9,}      {fp:>9,}")
print(f"  Actual 1 (FN/TP)  {fn:>9,}      {tp:>9,}")
print()
print(f"Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1        : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}  <-- meaningless here")
print()
print("Baselines:")
print(f"  All-zero model: accuracy 76.01%, recall 0.00%")
print(f"  All-one model : precision 23.99%, recall 100%")

Confusion matrix (threshold=0.5):
                   Predicted 0    Predicted 1
  Actual 0 (TN/FP)      7,272          7,931
  Actual 1 (FN/TP)      2,312          2,485

Precision : 0.2386
Recall    : 0.5180
F1        : 0.3267
Accuracy  : 0.4879  <-- meaningless here

Baselines:
  All-zero model: accuracy 76.01%, recall 0.00%
  All-one model : precision 23.99%, recall 100%


In [10]:
thresholds = [0.3, 0.5, 0.7]
rows = []
for t in thresholds:
    pred = (y_proba >= t).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, pred).ravel()
    rows.append({
        'threshold': t,
        'flagged': int(tp_t + fp_t),
        'TP': int(tp_t), 'FP': int(fp_t), 'FN': int(fn_t), 'TN': int(tn_t),
        'precision': round(precision_score(y_test, pred, zero_division=0), 4),
        'recall': round(recall_score(y_test, pred, zero_division=0), 4),
        'f1': round(f1_score(y_test, pred, zero_division=0), 4),
    })

threshold_df = pd.DataFrame(rows)
print("Threshold trade-off — logistic regression:")
print(threshold_df.to_string(index=False))
print()
print("Diagnostic: does precision stay at ~0.2399 (base rate) across thresholds?")
print("If yes → the model has no discriminative power at any threshold.")

Threshold trade-off — logistic regression:
 threshold  flagged   TP    FP   FN    TN  precision  recall     f1
       0.3    20000 4797 15203    0     0     0.2399   1.000 0.3869
       0.5    10416 2485  7931 2312  7272     0.2386   0.518 0.3267
       0.7        0    0     0 4797 15203     0.0000   0.000 0.0000

Diagnostic: does precision stay at ~0.2399 (base rate) across thresholds?
If yes → the model has no discriminative power at any threshold.


In [11]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['Customer_ID']))

X_tr_g = X.iloc[train_idx];  X_te_g = X.iloc[test_idx]
y_tr_g = y.iloc[train_idx];  y_te_g = y.iloc[test_idx]

print("Group split (by Customer_ID):")
print(f"  Train rows      : {len(X_tr_g):,}")
print(f"  Test rows       : {len(X_te_g):,}")
print(f"  Train customers : {df.iloc[train_idx]['Customer_ID'].nunique()}")
print(f"  Test customers  : {df.iloc[test_idx]['Customer_ID'].nunique()}")
overlap = len(set(df.iloc[train_idx]['Customer_ID']) & set(df.iloc[test_idx]['Customer_ID']))
print(f"  Overlap         : {overlap}  (must be 0)")
print(f"  Test fraud rate : {round(y_te_g.mean(), 4)}")

model_g = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model_g.fit(X_tr_g, y_tr_g)
y_pred_g = model_g.predict(X_te_g)

print()
print("Group-split metrics @ threshold 0.5:")
print(f"  Precision : {precision_score(y_te_g, y_pred_g, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(y_te_g, y_pred_g, zero_division=0):.4f}")
print(f"  F1        : {f1_score(y_te_g, y_pred_g, zero_division=0):.4f}")

Group split (by Customer_ID):
  Train rows      : 80,076
  Test rows       : 19,924
  Train customers : 517
  Test customers  : 130
  Overlap         : 0  (must be 0)
  Test fraud rate : 0.2394

Group-split metrics @ threshold 0.5:
  Precision : 0.2433
  Recall    : 0.5919
  F1        : 0.3449


In [12]:
from sklearn.ensemble import RandomForestClassifier
import time

start = time.time()
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
print(f"Trained in {time.time() - start:.2f} seconds.")

Trained in 3.15 seconds.


In [13]:
y_proba_rf = rf.predict_proba(X_test)[:, 1]
y_pred_rf = (y_proba_rf >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_rf).ravel()
print("Random forest @ threshold 0.5:")
print(f"  TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}")
print()
print(f"Precision : {precision_score(y_test, y_pred_rf, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_rf, zero_division=0):.4f}")
print(f"F1        : {f1_score(y_test, y_pred_rf, zero_division=0):.4f}")
print()
print(f"Compare to logistic: precision 0.2386, recall 0.5180, F1 0.3267")
print(f"Base rate baseline : precision 0.2399, recall 1.0000")

Random forest @ threshold 0.5:
  TN=8,579  FP=6,624  FN=2,714  TP=2,083

Precision : 0.2392
Recall    : 0.4342
F1        : 0.3085

Compare to logistic: precision 0.2386, recall 0.5180, F1 0.3267
Base rate baseline : precision 0.2399, recall 1.0000


In [14]:
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)

importance_df['importance'] = importance_df['importance'].round(4)
print("Top 15 feature importances:")
print(importance_df.head(15).to_string(index=False))
print()
print(f"Sum of top 5 importances : {importance_df.head(5)['importance'].sum():.4f}")
print(f"Sum of top 10 importances: {importance_df.head(10)['importance'].sum():.4f}")
print(f"Uniform baseline (1/43)  : {1/43:.4f}")

Top 15 feature importances:
                     feature  importance
                  amount_log      0.1901
Transaction_Amount_Deviation      0.1890
                        hour      0.1291
                 day_of_week      0.0743
 Days_Since_Last_Transaction      0.0599
       Transaction_Frequency      0.0434
                  is_weekend      0.0155
  Transaction_Channel_Mobile      0.0126
  Transaction_Status_Pending      0.0117
Transaction_Channel_In-store      0.0116
  Transaction_Channel_Online      0.0115
Transaction_Status_Completed      0.0110
           Device_OS_Windows      0.0110
           Device_OS_Android      0.0109
   Transaction_Status_Failed      0.0107

Sum of top 5 importances : 0.6424
Sum of top 10 importances: 0.7372
Uniform baseline (1/43)  : 0.0233


In [16]:
import io, contextlib, textwrap
from PIL import Image, ImageDraw, ImageFont

def render_text_to_png(text, out_path, font_size=17, wrap_width=115):
    wrapped = []
    for line in text.split("\n"):
        if len(line) <= wrap_width:
            wrapped.append(line)
        else:
            wrapped.extend(textwrap.wrap(line, width=wrap_width,
                                         subsequent_indent="    ",
                                         break_long_words=True,
                                         break_on_hyphens=False))
    font = ImageFont.truetype(r"C:\Windows\Fonts\consola.ttf", font_size)
    tmp = Image.new("RGB", (10, 10))
    bbox = ImageDraw.Draw(tmp).textbbox((0, 0), "M" * 20, font=font)
    cw = (bbox[2] - bbox[0]) / 20
    lh, pad = 23, 28
    width = int(max(len(l) for l in wrapped) * cw) + pad * 2
    height = len(wrapped) * lh + pad * 2
    img = Image.new("RGB", (width, height), (30, 30, 30))
    d = ImageDraw.Draw(img)
    y = pad
    for line in wrapped:
        d.text((pad, y), line, fill=(225, 225, 225), font=font)
        y += lh
    img.save(out_path)
    print(f"Saved: {out_path}  ({img.size[0]} x {img.size[1]} px)")

# --- Screenshot 1: threshold table ---
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    print("Logistic Regression - Threshold Trade-off (null result)\n")
    print("Test set: 20,000 rows, stratified 24% fraud")
    print("Model: LogisticRegression, class_weight='balanced'\n")
    print(threshold_df.to_string(index=False))
    print("\nDiagnostic: precision is 0.2399 at threshold 0.3 (flagging every row),")
    print("0.2386 at 0.5, and 0.0000 at 0.7. Base rate = 0.2399.")
    print("Precision equals base rate at every threshold = model has no")
    print("discriminative power. Recall 1.0 at 0.3 is arithmetic: flagging")
    print("all rows catches all frauds by construction.")
    print("\nGroup-split check (by Customer_ID):")
    print("  Train: 517 customers | Test: 130 customers | Overlap: 0")
    print("  Precision 0.2433 | Recall 0.5919 | F1 0.3449")
    print("  Same null. Not caused by customer memorisation.")
raw = buf.getvalue()
print(raw)
render_text_to_png(raw, r"C:\Users\boses\Desktop\bfsi-fraud-project\outputs\04_lr_threshold_table.png")

# --- Screenshot 2: feature importance ---
buf2 = io.StringIO()
with contextlib.redirect_stdout(buf2):
    print("Random Forest - Top 15 Feature Importances (artifact, not signal)\n")
    print(importance_df.head(15).to_string(index=False))
    print()
    print(f"Sum of top 5 importances : {importance_df.head(5)['importance'].sum():.4f}")
    print(f"Sum of top 10 importances: {importance_df.head(10)['importance'].sum():.4f}")
    print(f"Uniform baseline (1/43)  : {1/43:.4f}")
    print()
    print("Why concentration here is a trap:")
    print("  - amount_log (0.19) + Transaction_Amount_Deviation (0.19) measure")
    print("    the same underlying quantity. Correlated features split noise")
    print("    evenly across both. One feature's worth of signal, not two.")
    print("  - hour (0.13) has uniform distribution (4,046-4,351 per hour in")
    print("    Block A) but 24 possible values give the tree 23 split points,")
    print("    each reducing impurity marginally. Used, not informative.")
    print("  - Definitive test: RF precision 0.2392 vs base rate 0.2399.")
    print("    Indistinguishable. If top-5 features carried real signal,")
    print("    precision would exceed base rate by a visible margin.")
raw2 = buf2.getvalue()
print(raw2)
render_text_to_png(raw2, r"C:\Users\boses\Desktop\bfsi-fraud-project\outputs\05_feature_importance.png")

Logistic Regression - Threshold Trade-off (null result)

Test set: 20,000 rows, stratified 24% fraud
Model: LogisticRegression, class_weight='balanced'

 threshold  flagged   TP    FP   FN    TN  precision  recall     f1
       0.3    20000 4797 15203    0     0     0.2399   1.000 0.3869
       0.5    10416 2485  7931 2312  7272     0.2386   0.518 0.3267
       0.7        0    0     0 4797 15203     0.0000   0.000 0.0000

Diagnostic: precision is 0.2399 at threshold 0.3 (flagging every row),
0.2386 at 0.5, and 0.0000 at 0.7. Base rate = 0.2399.
Precision equals base rate at every threshold = model has no
discriminative power. Recall 1.0 at 0.3 is arithmetic: flagging
all rows catches all frauds by construction.

Group-split check (by Customer_ID):
  Train: 517 customers | Test: 130 customers | Overlap: 0
  Precision 0.2433 | Recall 0.5919 | F1 0.3449
  Same null. Not caused by customer memorisation.

Saved: C:\Users\boses\Desktop\bfsi-fraud-project\outputs\04_lr_threshold_table.png  (6

In [17]:
import os, re

readme_path = r"C:\Users\boses\Desktop\bfsi-fraud-project\README.md"
with open(readme_path, 'r', encoding='utf-8') as f:
    content = f.read()

new_section = """## Preliminary Findings (Python Modeling)

Three models, all null. The dataset has no learnable signal in the 43 engineered features.

**Setup.** Numeric: Transaction_Frequency, Transaction_Amount_Deviation, Days_Since_Last_Transaction, amount_log, hour, day_of_week, is_weekend. Categorical (one-hot): Transaction_Type, Payment_Gateway, Merchant_Category, Transaction_Channel, Device_OS, Transaction_Status. 43 features total. Stratified 80/20 split. class_weight='balanced'.

**Logistic regression (linear).** Confusion @ 0.5: TN=7,272, FP=7,931, FN=2,312, TP=2,485. Precision 0.2386, recall 0.5180, F1 0.3267. Accuracy 0.4879 — worse than all-zero baseline (0.7601). Precision equals base rate (0.2399) at every threshold.

| Threshold | Flagged | Precision | Recall | F1 |
|---|---|---|---|---|
| 0.3 | 20,000 (all) | 0.2399 | 1.000 | 0.3869 |
| 0.5 | 10,416 | 0.2386 | 0.518 | 0.3267 |
| 0.7 | 0 | 0.0000 | 0.000 | 0.0000 |

The 0.3 row — flagging every row — produces precision exactly equal to the base rate. The model's ranking ability is nil.

**Group split.** By Customer_ID: 517 train customers, 130 test, overlap 0. Precision 0.2433, recall 0.5919, F1 0.3449. Same null. Not caused by customer memorisation.

**Random forest (nonlinear).** Confusion @ 0.5: TN=8,579, FP=6,624, FN=2,714, TP=2,083. Precision 0.2392, recall 0.4342, F1 0.3085. Same null at the nonlinear model class.

**Feature importance — the false lead.** Top 5 (amount_log 0.19, Transaction_Amount_Deviation 0.19, hour 0.13, day_of_week 0.07, Days_Since_Last_Transaction 0.06) carry 0.64 of total importance, 5.5x uniform baseline. Not signal: amount_log + Transaction_Amount_Deviation are correlated (same underlying quantity, split noise); hour is uniform (4,046-4,351 per hour) but offers 23 split points. The definitive test is precision vs. base rate: 0.2392 vs. 0.2399 — indistinguishable.

**Why this matters.** Three independent signals — velocity (time), device concentration (network), behavioral columns (features) — all null. The synthetic generator drew each row independently. No feature predicts the label. The 24% fraud rate is a label rate, not a learnable target.

**The workflow is correct.** On real UPI data, the same pipeline — velocity query, device concentration, logistic regression, random forest, threshold table, precision-recall — surfaces the signals production fraud systems rely on. Queries, models, and memos preserved here as templates.

"""

# Insert before "## Status" section
pattern = r"(## Status)"
replacement = new_section + r"\1"
if "## Preliminary Findings (Python Modeling)" in content:
    print("Section already present. Skipping insert.")
else:
    content = re.sub(pattern, replacement, content, count=1)
    with open(readme_path, 'w', encoding='utf-8', newline='\n') as f:
        f.write(content)
    print("README updated with Python Modeling section.")

# Update Status checklist
content = open(readme_path, 'r', encoding='utf-8').read()
status_old = """- [x] Data inspection (Nov 3)
- [x] SQL layer — import verification, velocity, device concentration (Nov 5)
- [ ] Python modeling (Day 3 — next)
- [ ] Power BI dashboard
- [ ] One-page insight memo"""

status_new = """- [x] Data inspection (Nov 3)
- [x] SQL layer — velocity (null), device concentration (null) (Nov 5)
- [x] Python modeling — logistic + random forest, precision = base rate (Day 3)
- [ ] Power BI dashboard (Day 4)
- [ ] One-page insight memo (Day 5-6)"""

if status_old in content:
    content = content.replace(status_old, status_new)
    with open(readme_path, 'w', encoding='utf-8', newline='\n') as f:
        f.write(content)
    print("Status checklist updated.")
else:
    print("Status block not found in expected form. Check README manually.")

print("File size:", os.path.getsize(readme_path), "bytes")

README updated with Python Modeling section.
Status block not found in expected form. Check README manually.
File size: 7363 bytes


In [18]:
readme_path = r"C:\Users\boses\Desktop\bfsi-fraud-project\README.md"
with open(readme_path, 'r', encoding='utf-8') as f:
    content = f.read()

start = content.find("## Status")
print("=== Current Status section (verbatim) ===")
print(repr(content[start:start+400]))
print()
print("=== Rendered ===")
print(content[start:start+400])

=== Current Status section (verbatim) ===
'## Status\n\n- [x] Data inspection (Nov 3)\n- [x] SQL layer - import verification, velocity, device concentration (Nov 5)\n- [ ] Python modeling (Day 3 - next)\n- [ ] Power BI dashboard\n- [ ] One-page insight memo\n'

=== Rendered ===
## Status

- [x] Data inspection (Nov 3)
- [x] SQL layer - import verification, velocity, device concentration (Nov 5)
- [ ] Python modeling (Day 3 - next)
- [ ] Power BI dashboard
- [ ] One-page insight memo



In [19]:
readme_path = r"C:\Users\boses\Desktop\bfsi-fraud-project\README.md"
with open(readme_path, 'r', encoding='utf-8') as f:
    content = f.read()

old_status = """## Status

- [x] Data inspection (Nov 3)
- [x] SQL layer - import verification, velocity, device concentration (Nov 5)
- [ ] Python modeling (Day 3 - next)
- [ ] Power BI dashboard
- [ ] One-page insight memo"""

new_status = """## Status

- [x] Data inspection (Nov 3)
- [x] SQL layer - velocity null, device concentration null (Nov 5)
- [x] Python modeling - logistic + random forest, precision = base rate (Day 3)
- [ ] Power BI dashboard (Day 4)
- [ ] One-page insight memo (Day 5-6)"""

if old_status in content:
    content = content.replace(old_status, new_status)
    with open(readme_path, 'w', encoding='utf-8', newline='\n') as f:
        f.write(content)
    print("Status block replaced.")
else:
    print("Still no match — pasting the current block below so we can see exact whitespace:")
    start = content.find("## Status")
    print(repr(content[start:]))

# Verify
with open(readme_path, 'r', encoding='utf-8') as f:
    content = f.read()
start = content.find("## Status")
print()
print("=== New Status section ===")
print(content[start:])
print()
print("File size:", len(content), "bytes")

Status block replaced.

=== New Status section ===
## Status

- [x] Data inspection (Nov 3)
- [x] SQL layer - velocity null, device concentration null (Nov 5)
- [x] Python modeling - logistic + random forest, precision = base rate (Day 3)
- [ ] Power BI dashboard (Day 4)
- [ ] One-page insight memo (Day 5-6)


File size: 7395 bytes


In [20]:
readme_path = r"C:\Users\boses\Desktop\bfsi-fraud-project\README.md"
with open(readme_path, 'r', encoding='utf-8') as f:
    content = f.read()

print("Python Modeling section present:", "## Preliminary Findings (Python Modeling)" in content)
print("SQL Findings section present    :", "## Preliminary Findings (SQL Layer)" in content)
print("Threshold table present         :", "Precision equals base rate" in content or "precision equals base rate" in content)
print()
print("Section order in file:")
import re
for m in re.finditer(r"^## .+$", content, re.MULTILINE):
    print("  " + m.group(0))

Python Modeling section present: True
SQL Findings section present    : True
Threshold table present         : True

Section order in file:
  ## TL;DR
  ## Data
  ## Methods
  ## Repository Map
  ## Preliminary Findings (SQL Layer)
  ## Preliminary Findings (Python Modeling)
  ## Status


In [21]:
import os

artifact_dir = r"C:\Users\boses\Desktop\bfsi-fraud-project\outputs"
os.makedirs(artifact_dir, exist_ok=True)

threshold_csv = os.path.join(artifact_dir, 'threshold_table.csv')
importance_csv = os.path.join(artifact_dir, 'feature_importance.csv')

threshold_df.to_csv(threshold_csv, index=False)
importance_df.to_csv(importance_csv, index=False)

print("Saved:")
for p in [threshold_csv, importance_csv]:
    print(f"  {os.path.basename(p)}: {os.path.getsize(p)} bytes")

Saved:
  threshold_table.csv: 180 bytes
  feature_importance.csv: 1471 bytes


In [22]:
readme_path = r"C:\Users\boses\Desktop\bfsi-fraud-project\README.md"
with open(readme_path, 'r', encoding='utf-8') as f:
    content = f.read()

start = content.find("## Status")
print("=== Current Status section ===")
print(content[start:])
print()
print("Python Modeling section present:", "## Preliminary Findings (Python Modeling)" in content)
print("SQL Findings section present    :", "## Preliminary Findings (SQL Layer)" in content)

=== Current Status section ===
## Status

- [x] Data inspection (Nov 3)
- [x] SQL layer - velocity null, device concentration null (Nov 5)
- [x] Python modeling - logistic + random forest, precision = base rate (Day 3)
- [ ] Power BI dashboard (Day 4)
- [ ] One-page insight memo (Day 5-6)


Python Modeling section present: True
SQL Findings section present    : True


In [23]:
readme_path = r"C:\Users\boses\Desktop\bfsi-fraud-project\README.md"
with open(readme_path, 'r', encoding='utf-8') as f:
    content = f.read()

print("Python Modeling section present:", "## Preliminary Findings (Python Modeling)" in content)
print("Status shows 3 [x]            :", content.count("- [x]") >= 3)
print()
start = content.find("## Status")
print("=== Current Status section ===")
print(content[start:])

Python Modeling section present: True
Status shows 3 [x]            : True

=== Current Status section ===
## Status

- [x] Data inspection (Nov 3)
- [x] SQL layer - velocity null, device concentration null (Nov 5)
- [x] Python modeling - logistic + random forest, precision = base rate (Day 3)
- [ ] Power BI dashboard (Day 4)
- [ ] One-page insight memo (Day 5-6)



In [24]:
import os

readme_path = r"C:\Users\boses\Desktop\bfsi-fraud-project\README.md"
with open(readme_path, 'r', encoding='utf-8') as f:
    content = f.read()

# Read the dashboard link if saved
link_path = r"C:\Users\boses\Desktop\bfsi-fraud-project\dashboard\dashboard_link.txt"
if os.path.exists(link_path):
    with open(link_path, 'r', encoding='utf-8') as f:
        link = f.read().strip()
else:
    link = "(Live report link pending — screenshot available below)"

dashboard_section = f"""## Dashboard

**Power BI report:** {link}

**Screenshot:** [`outputs/06_dashboard_screenshot.png`](outputs/06_dashboard_screenshot.png)

The dashboard visualises the three null results so a reader can verify the finding in ten seconds:

- **Precision vs. Recall across thresholds** — precision stays flat at the base rate (0.2399) while recall falls from 1.0 to 0. Visual proof of no ranking ability.
- **Feature importance vs. uniform baseline** — the top-5 features tower above 1/43, but the subtitle explains why concentration is a Gini artifact, not signal: precision equals base rate (0.2392 vs 0.2399).
- **Fraud rate by hour of day** — 24 bars all at roughly 0.24, no cycle. Uniform distribution visualised.
- **Data quality diagnostic** — 100,000 transactions, 0 missing values, 647 customers, 647 devices (1:1 mapping), 99,998 unique IPs.

The dashboard is a template. On genuine UPI data, the precision-recall chart would show precision climbing steeply with threshold, the feature-importance chart would name real signal-carrying columns, and the hour-of-day chart would show fraud concentrating in low-traffic hours.

"""

# Insert Dashboard section before "## Status"
if "## Dashboard" not in content:
    start = content.find("## Status")
    if start == -1:
        content = content + "\n" + dashboard_section
    else:
        content = content[:start] + dashboard_section + content[start:]
    print("Dashboard section inserted.")

# Update Status checklist
old_status = "- [ ] Power BI dashboard (Day 4)"
new_status = "- [x] Power BI dashboard - published, null results visualised (Day 4)"
if old_status in content:
    content = content.replace(old_status, new_status)
    print("Status updated.")
else:
    print("Status line not found. Check current Status section.")

with open(readme_path, 'w', encoding='utf-8', newline='\n') as f:
    f.write(content)

print("File size:", os.path.getsize(readme_path), "bytes")

Dashboard section inserted.
Status updated.
File size: 8639 bytes
